# Polymarket API Python

**Resources:**
- [Polymarket API Docs](https://docs.polymarket.com/)
- [Gamma API (Market Data)](https://gamma-api.polymarket.com/)
- [Data API (User Data)](https://data-api.polymarket.com/)
- [py-clob-client (3rd Party using Official SDK)](https://github.com/Polymarket/py-clob-client)

## 1. Setup

In [ ]:
# pip install py-clob-client requests

In [3]:
import requests
import time
import json
from pprint import pprint

from py_clob_client.client import ClobClient
from py_clob_client.clob_types import OrderArgs, MarketOrderArgs, OrderType, OpenOrderParams, BalanceAllowanceParams, AssetType
from py_clob_client.order_builder.constants import BUY, SELL

GAMMA_API = "https://gamma-api.polymarket.com"
DATA_API = "https://data-api.polymarket.com"
CLOB_API = "https://clob.polymarket.com"

## 2. Market Discovery

In [ ]:
# Fetch active markets sorted by volume
response = requests.get(
    f"{GAMMA_API}/markets",
    params={
        "limit": 10,
        "active": True,
        "closed": False,
        "order": "volume24hr",
        "ascending": False
    }
)
markets = response.json()
print(f"Found {len(markets)} markets\n")

In [ ]:
for m in markets[:5]:
    print(f"Question: {m['question']}")
    print(f"  Volume 24h: ${m.get('volume24hr', 0):,.0f}")
    print(f"  Liquidity: ${m.get('liquidityNum', 0):,.0f}")
    print(f"  Prices: {m.get('outcomePrices', 'N/A')}")
    print()

## 3. Market Deep Dive

In [ ]:
market = markets[1]
market

### Create a special Market from Slug

In [9]:
### Create a special Market
import requests

slug = "elon-musk-of-tweets-january-26-january-28-90-114"

market_url = f"{GAMMA_API}/markets/slug/{slug}"

response = requests.get(market_url)
if response.status_code != 200:
    raise ValueError(f"API error {response.status_code}: {response.text[:200]}")

try:
    market = response.json()
except Exception:
    raise ValueError(f"Non-JSON response: {response.text[:200]}")

# print(market)
market

{'id': '1259141',
 'question': 'Will Elon Musk post 90-114 tweets from January 26 to January 28, 2026?',
 'conditionId': '0x81cf3f03aa3f2a297485df5855df13364628b07823fd27f71077d88d42233a86',
 'slug': 'elon-musk-of-tweets-january-26-january-28-90-114',
 'resolutionSource': 'https://x.com/elonmusk',
 'endDate': '2026-01-28T17:00:00Z',
 'liquidity': '5879.0793',
 'startDate': '2026-01-24T17:12:38.900129Z',
 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/elon-musk-of-tweets-nov-22-29-apMPG21-pzx_.jpg',
 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/elon-musk-of-tweets-nov-22-29-apMPG21-pzx_.jpg',
 'description': 'This market will resolve according to the number of times Elon Musk (@elonmusk), posts on X from January 26 12:00 PM ET to January 28, 2026 12:00 PM ET.\n\nFor the purposes of this market, only main feed posts, quote posts and reposts will count.\n\nReplies will NOT count towards the total - however, replies on the main feed such as https://x.com/el

In [10]:
print(f"Market: {market['question']}")
print(f"End Date: {market['endDate']}")
print(f"Condition ID: {market['conditionId']}")

Market: Will Elon Musk post 90-114 tweets from January 26 to January 28, 2026?
End Date: 2026-01-28T17:00:00Z
Condition ID: 0x81cf3f03aa3f2a297485df5855df13364628b07823fd27f71077d88d42233a86


In [11]:
clob_token_ids = market.get('clobTokenIds')
clob_token_ids = json.loads(clob_token_ids)
print(f"Token IDs: {clob_token_ids}")

if len(clob_token_ids) >= 2:
    yes_token_id = clob_token_ids[0]
    no_token_id = clob_token_ids[1]
    print(f"YES token: {yes_token_id}")
    print(f"NO token: {no_token_id}")

Token IDs: ['50214305297176962065773245512961547550198516374144624237468290586352509598213', '35441461644888271545257282000214276514361253286421846140522153842517104943246']
YES token: 50214305297176962065773245512961547550198516374144624237468290586352509598213
NO token: 35441461644888271545257282000214276514361253286421846140522153842517104943246


## 4. Order Book Analysis

In [12]:
# Read-only client for order book
client = ClobClient(CLOB_API)

In [13]:
# Fetch order book for YES token
if yes_token_id:
    book = client.get_order_book(yes_token_id)
    
    sorted_bids = sorted(book.bids, key=lambda x: float(x.price), reverse=True)
    sorted_asks = sorted(book.asks, key=lambda x: float(x.price), reverse=False)
    
    print("=== YES Token Order Book ===")
    print(f"\nTop 5 Asks (sell orders):")
    for ask in sorted_asks[:5]:
        print(f"  Price: {ask.price} | Size: {ask.size}")
        
    print(f"\nTop 5 Bids (buy orders):")
    for bid in sorted_bids[:5]:
        print(f"  Price: {bid.price} | Size: {bid.size}")


=== YES Token Order Book ===

Top 5 Asks (sell orders):
  Price: 0.34 | Size: 183.61
  Price: 0.35 | Size: 233.23
  Price: 0.36 | Size: 274.19
  Price: 0.37 | Size: 97
  Price: 0.38 | Size: 505.02

Top 5 Bids (buy orders):
  Price: 0.33 | Size: 531.09
  Price: 0.32 | Size: 900.47
  Price: 0.31 | Size: 596.27
  Price: 0.3 | Size: 10
  Price: 0.27 | Size: 100


In [14]:
# Get midpoint, prices, and spread
mid = client.get_midpoint(yes_token_id)
buy_price = client.get_price(yes_token_id, side="BUY")
sell_price = client.get_price(yes_token_id, side="SELL")
spread = client.get_spread(yes_token_id)

print(f"Midpoint: {mid['mid']}")
print(f"Best ask (buy at): {buy_price['price']}")
print(f"Best bid (sell at): {sell_price['price']}")
print(f"Spread: {spread['spread']}")

Midpoint: 0.335
Best ask (buy at): 0.33
Best bid (sell at): 0.34
Spread: 0.01


## 5. Authentication

In [15]:
# Your credentials (keep these secret!)
FUNDER_ADDRESS = "0xC41997C65144683AB62051EDd1f80B034756E588"  # For proxy wallets
PRIVATE_KEY = "0xfb9e13860aa141e269e0b558e0f3600a7d81f2378b3da8c18572619012efab51"  # Never commit this

# Signature types:
# 0 = EOA (MetaMask, hardware wallet)
# 1 = Email/Magic wallet
# 2 = Browser wallet proxy
SIGNATURE_TYPE = 1

In [16]:
# Initialize authenticated client
auth_client = ClobClient(
    CLOB_API,
    key=PRIVATE_KEY,
    chain_id=137,
    signature_type=SIGNATURE_TYPE,
    funder=FUNDER_ADDRESS
)

# Get & Set API credentials
creds = auth_client.derive_api_key()
auth_client.set_api_creds(creds)

In [17]:
balance = auth_client.get_balance_allowance(BalanceAllowanceParams(asset_type=AssetType.COLLATERAL))
usdc_balance = int(balance['balance']) / 1e6
print(f"USDC Balance: ${usdc_balance:.2f}")

USDC Balance: $126.15


## 6. Place a Market Order

In [ ]:
# Create a market order: Spend $5 on YES shares at market price
market_order = MarketOrderArgs(
    token_id=yes_token_id,
    amount=5.0,  # Dollar amount to spend
    side=BUY,
    order_type=OrderType.FOK  # Fill-or-Kill
)

signed_market_order = auth_client.create_market_order(market_order)
print("Market order signed!")

In [ ]:
# Execute market order
response = auth_client.post_order(signed_market_order, OrderType.FOK)
print("Market order executed!")
pprint(response)

## 7. Place a Limit Order

In [ ]:
# Create a limit order: Buy 10 YES shares at $0.50
limit_order = OrderArgs(
    token_id=yes_token_id,
    price=0.1,  # Price per share
    size=10.0,   # Number of shares
    side=BUY
)

# Sign the order
signed_order = auth_client.create_order(limit_order)
print("Order signed!")

Order signed!
SignedOrder(order=<py_order_utils.model.order.Order object at 0x000001E44342C6D0>,
            signature='0xa16b58217e85c3765d9980db86b15f8e69c2096e6683785b519136ad0a9082737b84cd8790487f2edf2b677c9b8610644e863f4ca35c4177c9eb56723e1f854b1b')


In [ ]:
# Post the order (GTC = Good Till Cancelled)
response = auth_client.post_order(signed_order, OrderType.GTC)
print(f"Order placed!")
pprint(response)

## 8. Open Orders

In [ ]:
# Get your open orders
open_orders = auth_client.get_orders(OpenOrderParams())
print(f"Open orders: {len(open_orders)}")

for order in open_orders[:5]:
    print(f"  ID: {order['id'][:20]}...")
    print(f"  Side: {order['side']} | Price: {order['price']} | Size: {order['original_size']}")
    print()

## 9. Cancel Orders

In [ ]:
# Cancel a specific order
open_orders = auth_client.get_orders(OpenOrderParams())
if open_orders:
    order_id = open_orders[0]["id"]
    result = auth_client.cancel(order_id)
    print(f"Cancelled order: {order_id[:20]}...")
    pprint(result)

In [ ]:
# Cancel all orders
result = auth_client.cancel_all()
print("All orders cancelled!")
pprint(result)

## 10. BONUS 1: Price Tracker

In [ ]:
# default ticker

def track_price(token_id, duration_seconds=30, interval=5):
    """Track price changes in real-time."""
    print(f"Tracking price for {duration_seconds}s...\n")

    client = ClobClient(CLOB_API)
    start_time = time.time()
    prices = []

    while time.time() - start_time < duration_seconds:
        mid = client.get_midpoint(token_id)
        mid_price = float(mid['mid'])
        timestamp = time.strftime("%H:%M:%S")
        prices.append(mid_price)

        change = ""
        if len(prices) > 1:
            diff = prices[-1] - prices[-2]
            change = f" ({'+' if diff >= 0 else ''}{diff:.4f})"

        print(f"[{timestamp}] Price: {mid_price}{change}")
        time.sleep(interval)

    print(f"\nTotal change: {prices[-1] - prices[0]:.4f}")
    return prices

In [ ]:
# extended price ticker

from IPython.display import clear_output

def track_price(token_id, duration_seconds=30, interval=5):
    """Track price changes in real-time like a ticker."""
    client = ClobClient(CLOB_API)
    start_time = time.time()
    prices = []
    start_price = None

    while time.time() - start_time < duration_seconds:
        mid = client.get_midpoint(token_id)
        mid_price = float(mid['mid'])
        timestamp = time.strftime("%H:%M:%S")
        prices.append(mid_price)
        
        if start_price is None:
            start_price = mid_price

        # Berechne Änderungen
        step_change = ""
        if len(prices) > 1:
            diff = prices[-1] - prices[-2]
            step_change = f"Step: {'+' if diff >= 0 else ''}{diff:.4f}"
        
        total_change = prices[-1] - start_price
        total_str = f"Total: {'+' if total_change >= 0 else ''}{total_change:.4f}"
        
        elapsed = int(time.time() - start_time)
        remaining = duration_seconds - elapsed

        # Ticker-Ausgabe: Aktualisiert in derselben Zelle
        clear_output(wait=True)
        print(f"🔄 PRICE TRACKER | Remaining: {remaining}s")
        print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        print(f"[{timestamp}] Price: {mid_price:.4f}")
        print(f"{step_change}  |  {total_str}")
        
        time.sleep(interval)

    # Finale Ausgabe
    clear_output(wait=True)
    print(f"✅ TRACKING COMPLETE")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"Start Price:  {start_price:.4f}")
    print(f"Final Price:  {prices[-1]:.4f}")
    print(f"Total Change: {'+' if (prices[-1] - start_price) >= 0 else ''}{prices[-1] - start_price:.4f}")
    print(f"Samples:      {len(prices)}")
    return prices

In [ ]:
# Run the tracker
prices = track_price(yes_token_id, duration_seconds=10, interval=1)

## 11. BONUS 2: Address Position Tracker

In [ ]:
def get_user_positions(wallet_address):
    """Get a user's current positions."""
    url = f"{DATA_API}/positions"
    params = {"user": wallet_address}
    response = requests.get(url, params=params)
    return response.json()


In [ ]:
address = ""
positions = get_user_positions(address)
positions